# XYZ Batch Analysis for NPH / NPH_PISTON / NVT_PISTON

This notebook compares the batch XYZ outputs in:

- `results/20260309_NPH_batch_xyz_saving`
- `results/20260311_NPH_batch_xyz_saving_piston`
- `results/20260313_NVT_batch_xyz_saving_piston`

It does four things:

1. Reuses the existing particle snapshot plotting function to backfill the missing NPH-family frame plots, with the same 20-snapshot-per-phase cadence used by the run drivers.
2. Computes the second-phase order parameter and plots **three lines per temperature** (one line per simulation family).
3. Computes the second-phase average A-fraction profile along `x / Lx` and plots **three lines per temperature**. The y-axis is `1` for all-A and `0` for all-B.
4. Computes the second-phase volume for the two NPH families and plots it versus second-phase time.

## Order-parameter definition

For each saved second-phase frame, particles are binned in reduced coordinate `x / Lx`.
In bin `b` at time `t`,

- `f_A(b, t) = N_A(b, t) / (N_A(b, t) + N_B(b, t))`
- `phi(b, t) = |2 f_A(b, t) - 1| = |N_A(b, t) - N_B(b, t)| / (N_A(b, t) + N_B(b, t))`

The plotted order parameter is the mean over occupied x-bins,

- `Phi(t) = mean_b phi(b, t)`

so `Phi(t) = 0` means every occupied x-bin is locally mixed, and `Phi(t) = 1` means every occupied x-bin is single-species.

In [7]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "python" / "particle_csv.py").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root containing python/particle_csv.py")


REPO_ROOT = find_repo_root()
PYTHON_DIR = REPO_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

from particle_csv import plot_particle_csv

plt.rcParams["figure.figsize"] = (8.0, 4.5)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.bbox"] = "tight"

PHASE_SETTINGS = {
    "NVT": {"dt": 1.0e-3, "total_steps": 100_000, "plots": 20},
    "NPH": {"dt": 1.0e-4, "total_steps": 5_000_000, "plots": 20},
    "NPH_PISTON": {"dt": 1.0e-4, "total_steps": 5_000_000, "plots": 20},
    "NVT_100": {"dt": 1.0e-3, "total_steps": 100_000, "plots": 20},
    "NVT_500": {"dt": 1.0e-4, "total_steps": 5_000_000, "plots": 20},
}

RESULT_SPECS = [
    {
        "id": "nph",
        "label": "20260309 NPH",
        "color": "#1f77b4",
        "root": REPO_ROOT / "results" / "20260309_NPH_batch_xyz_saving",
        "trajectory_name": "trajectory.xyz",
        "config_name": "config_large.json",
        "phase_order": ("NVT", "NPH"),
        "second_phase": "NPH",
        "snapshot_dirs": {"NVT": "NVT", "NPH": "NPH"},
        "snapshot_csv": {"NVT": "nvt_plot_input.csv", "NPH": "nph_plot_input.csv"},
        "plot_snapshots": True,
        "has_volume": True,
    },
    # {
    #     "id": "nph_piston",
    #     "label": "20260311 NPH piston",
    #     "color": "#d62728",
    #     "root": REPO_ROOT / "results" / "20260311_NPH_batch_xyz_saving_piston",
    #     "trajectory_name": "trajectory_piston.xyz",
    #     "config_name": "config_large_piston.json",
    #     "phase_order": ("NVT", "NPH_PISTON"),
    #     "second_phase": "NPH_PISTON",
    #     "snapshot_dirs": {"NVT": "NVT", "NPH_PISTON": "NPH_PISTON"},
    #     "snapshot_csv": {"NVT": "nvt_plot_input.csv", "NPH_PISTON": "nph_piston_plot_input.csv"},
    #     "plot_snapshots": True,
    #     "has_volume": True,
    # },
    {
        "id": "nvt_piston",
        "label": "20260313 NVT piston",
        "color": "#2ca02c",
        "root": REPO_ROOT / "results" / "20260313_NVT_batch_xyz_saving_piston",
        "trajectory_name": "trajectory_nvt.xyz",
        "config_name": "config_large_piston.json",
        "phase_order": ("NVT_100", "NVT_500"),
        "second_phase": "NVT_500",
        "snapshot_dirs": {"NVT_100": "NVT_100", "NVT_500": "NVT_500"},
        "snapshot_csv": {"NVT_100": "nvt_100_plot_input.csv", "NVT_500": "nvt_500_plot_input.csv"},
        "plot_snapshots": False,
        "has_volume": False,
    },
]

CACHE_DIRNAME = "analysis_xyz_cache"
N_X_BINS = 200
TARGET_A_CENTER_BIN = N_X_BINS // 2
FIG_DPI = 150
SNAPSHOT_DPI = 110

ANALYSIS_ROOT = REPO_ROOT / "results" / "analysis_NPH_xyz_batches"
ORDER_DIR = ANALYSIS_ROOT / "order_parameter"
PROFILE_DIR = ANALYSIS_ROOT / "density_x"
VOLUME_DIR = ANALYSIS_ROOT / "volume"

for path in (ANALYSIS_ROOT, ORDER_DIR, PROFILE_DIR, VOLUME_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Comparison figures will be written under: {ANALYSIS_ROOT}")

Repo root: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA
Comparison figures will be written under: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA/results/analysis_NPH_xyz_batches


In [2]:
TEMP_PATTERN = re.compile(r"^T=(?P<value>[-+]?\d*\.?\d+)$")


def parse_temperature_dir_name(path: Path) -> float:
    match = TEMP_PATTERN.match(path.name)
    if not match:
        raise ValueError(f"Temperature directory must look like T=<value>, got {path.name!r}")
    return float(match.group("value"))


def safe_temp_label(temp_value: float) -> str:
    return f"{temp_value:.1f}".replace(".", "p")


def list_temperature_dirs(spec: dict) -> dict[float, Path]:
    mapping: dict[float, Path] = {}
    for temp_dir in sorted(spec["root"].glob("T=*")):
        if temp_dir.is_dir():
            mapping[parse_temperature_dir_name(temp_dir)] = temp_dir
    return mapping


def shared_temperature_values(specs: list[dict]) -> list[float]:
    per_spec_values = [set(list_temperature_dirs(spec).keys()) for spec in specs]
    if not per_spec_values:
        return []
    shared = sorted(set.intersection(*per_spec_values))
    return shared


def load_config(spec: dict, temp_dir: Path) -> dict:
    config_path = temp_dir / "configs" / spec["config_name"]
    with config_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def parse_metadata_line(line: str) -> dict[str, float | int | str]:
    parsed: dict[str, float | int | str] = {}
    for token in line.strip().split():
        key, value = token.split("=", 1)
        if key == "phase":
            parsed[key] = value
        elif key in {"global_step", "phase_step"}:
            parsed[key] = int(value)
        else:
            parsed[key] = float(value)
    return parsed


def expected_snapshot_targets(spec: dict, temp_dir: Path) -> dict[str, dict[int, dict]]:
    frames_root = temp_dir / "frames"
    csv_root = frames_root / "csv"
    csv_root.mkdir(parents=True, exist_ok=True)

    expected: dict[str, dict[int, dict]] = {}
    global_offset = 0
    for phase_name in spec["phase_order"]:
        phase_cfg = PHASE_SETTINGS[phase_name]
        plot_interval = max(1, phase_cfg["total_steps"] // phase_cfg["plots"])
        phase_dir = frames_root / spec["snapshot_dirs"][phase_name]
        phase_dir.mkdir(parents=True, exist_ok=True)
        csv_path = csv_root / spec["snapshot_csv"][phase_name]
        expected[phase_name] = {}
        snapshot_idx = 0
        for phase_step in range(plot_interval, phase_cfg["total_steps"] + 1, plot_interval):
            snapshot_idx += 1
            global_step = global_offset + phase_step
            svg_path = phase_dir / f"snapshot_{snapshot_idx:02d}_step_{global_step}.svg"
            expected[phase_name][phase_step] = {
                "phase_step": phase_step,
                "global_step": global_step,
                "index": snapshot_idx,
                "path": svg_path,
                "csv_path": csv_path,
            }
        global_offset += phase_cfg["total_steps"]
    return expected


def missing_snapshot_paths(spec: dict, temp_dir: Path) -> list[Path]:
    if not spec["plot_snapshots"]:
        return []
    expected = expected_snapshot_targets(spec, temp_dir)
    missing: list[Path] = []
    for phase_map in expected.values():
        for target in phase_map.values():
            if not target["path"].exists():
                missing.append(target["path"])
    return missing


def parse_particle_block(lines: list[str], *, include_y: bool) -> tuple[np.ndarray, np.ndarray | None, np.ndarray]:
    n_rows = len(lines)
    x = np.empty(n_rows, dtype=np.float64)
    y = np.empty(n_rows, dtype=np.float64) if include_y else None
    types = np.empty(n_rows, dtype=np.int8)

    for idx, line in enumerate(lines):
        tag, x_str, y_str, _ = line.split()
        x[idx] = float(x_str)
        if include_y:
            assert y is not None
            y[idx] = float(y_str)
        types[idx] = 0 if tag == "A" else 1

    return x, y, types


def write_particle_csv_frame(
    csv_path: Path,
    x: np.ndarray,
    y: np.ndarray,
    types: np.ndarray,
    *,
    box_w: float,
    box_h: float,
    sigma_aa: float,
    sigma_bb: float,
) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        handle.write(
            f"box_w,{box_w:.12f},box_h,{box_h:.12f},sigma_aa,{sigma_aa:.12f},"
            f"sigma_bb,{sigma_bb:.12f},draw_box,1,n_particles,{len(types)}\n"
        )
        for xi, yi, ti in zip(x, y, types):
            handle.write(f"x,{xi:.12f},y,{yi:.12f},type,{int(ti)}\n")


def composition_profile_from_frame(x: np.ndarray, types: np.ndarray, box_w: float) -> tuple[np.ndarray, np.ndarray, float]:
    reduced_x = np.mod(x / box_w, 1.0)
    bin_idx = np.floor(reduced_x * N_X_BINS).astype(np.int32)
    np.clip(bin_idx, 0, N_X_BINS - 1, out=bin_idx)

    total_counts = np.bincount(bin_idx, minlength=N_X_BINS)
    a_counts = np.bincount(bin_idx[types == 0], minlength=N_X_BINS)

    a_fraction = np.full(N_X_BINS, np.nan, dtype=np.float64)
    occupied = total_counts > 0
    a_fraction[occupied] = a_counts[occupied] / total_counts[occupied]

    if np.any(occupied):
        order_value = float(np.mean(np.abs(2.0 * a_fraction[occupied] - 1.0)))
    else:
        order_value = float("nan")

    return a_fraction, occupied, order_value


def alignment_shift_bins(signed_profile: np.ndarray) -> int:
    positive = np.clip(signed_profile, 0.0, None)
    if np.any(positive > 0.0):
        phase_angles = 2.0 * np.pi * (np.arange(signed_profile.size) + 0.5) / signed_profile.size
        vector = np.sum(positive * np.exp(1j * phase_angles))
        if abs(vector) > 1.0e-12:
            current_bin = ((np.angle(vector) % (2.0 * np.pi)) * signed_profile.size / (2.0 * np.pi)) - 0.5
        else:
            current_bin = float(np.argmax(signed_profile))
    else:
        current_bin = float(np.argmax(signed_profile))
    return int(np.round(TARGET_A_CENTER_BIN - current_bin))


def cache_path_for(temp_dir: Path) -> Path:
    cache_dir = temp_dir / CACHE_DIRNAME
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / "second_phase_summary.npz"


def load_cached_summary(cache_path: Path) -> dict:
    data = np.load(cache_path)
    return {
        "phase_time": data["phase_time"],
        "order_parameter": data["order_parameter"],
        "volume": data["volume"],
        "bin_centers": data["bin_centers"],
        "mean_a_fraction": data["mean_a_fraction"],
        "n_frames": int(data["n_frames"][0]),
    }

In [3]:
def analyze_temperature(
    spec: dict,
    temp_dir: Path,
    *,
    force_recompute: bool = False,
    force_snapshot_replot: bool = False,
) -> dict:
    cache_path = cache_path_for(temp_dir)
    missing_snapshots = missing_snapshot_paths(spec, temp_dir)
    need_full_pass = force_recompute or force_snapshot_replot or bool(missing_snapshots) or not cache_path.exists()

    if not need_full_pass:
        return load_cached_summary(cache_path)

    config = load_config(spec, temp_dir)
    sigma_aa = float(config["SIGMA_AA"])
    sigma_bb = float(config["SIGMA_BB"])
    expected_snapshots = expected_snapshot_targets(spec, temp_dir) if spec["plot_snapshots"] else {}

    second_phase_name = spec["second_phase"]
    phase_times: list[float] = []
    order_values: list[float] = []
    volumes: list[float] = []

    profile_sum = np.zeros(N_X_BINS, dtype=np.float64)
    profile_count = np.zeros(N_X_BINS, dtype=np.int64)

    trajectory_path = temp_dir / spec["trajectory_name"]
    with trajectory_path.open("r", encoding="utf-8") as handle:
        while True:
            header = handle.readline()
            if not header:
                break

            header = header.strip()
            if not header:
                continue

            n_particles = int(header)
            metadata = parse_metadata_line(handle.readline())
            phase_name = metadata["phase"]
            phase_step = int(metadata["phase_step"])
            wants_second_phase = phase_name == second_phase_name
            snapshot_target = expected_snapshots.get(phase_name, {}).get(phase_step)
            needs_snapshot = bool(
                spec["plot_snapshots"]
                and snapshot_target is not None
                and (force_snapshot_replot or not snapshot_target["path"].exists())
            )

            if not wants_second_phase and not needs_snapshot:
                for _ in range(n_particles):
                    skipped = handle.readline()
                    if not skipped:
                        raise RuntimeError(f"Unexpected end of file while skipping {trajectory_path}")
                continue

            particle_lines = [handle.readline() for _ in range(n_particles)]
            if not particle_lines or any(line == "" for line in particle_lines):
                raise RuntimeError(f"Unexpected end of file inside particle block for {trajectory_path}")

            x, y, types = parse_particle_block(particle_lines, include_y=needs_snapshot)

            if wants_second_phase:
                a_fraction, occupied, order_value = composition_profile_from_frame(
                    x,
                    types,
                    float(metadata["Lx"]),
                )
                signed_profile = np.zeros_like(a_fraction)
                signed_profile[occupied] = 2.0 * a_fraction[occupied] - 1.0
                shift = alignment_shift_bins(signed_profile)

                profile_sum += np.roll(np.nan_to_num(a_fraction, nan=0.0), shift)
                profile_count += np.roll(occupied.astype(np.int64), shift)

                phase_times.append(float(metadata["phase_time"]))
                order_values.append(order_value)
                volumes.append(float(metadata["Lx"]) * float(metadata["Ly"]))

            if needs_snapshot:
                assert y is not None
                write_particle_csv_frame(
                    snapshot_target["csv_path"],
                    x,
                    y,
                    types,
                    box_w=float(metadata["Lx"]),
                    box_h=float(metadata["Ly"]),
                    sigma_aa=sigma_aa,
                    sigma_bb=sigma_bb,
                )
                plot_particle_csv(
                    csv_path=snapshot_target["csv_path"],
                    output_path=snapshot_target["path"],
                    dpi=SNAPSHOT_DPI,
                    strict_box_limits=True,
                )

    if not phase_times:
        raise RuntimeError(
            f"No second-phase frames were found for {spec['label']} at {temp_dir.name}"
        )

    mean_a_fraction = np.divide(
        profile_sum,
        profile_count,
        out=np.full_like(profile_sum, np.nan, dtype=np.float64),
        where=profile_count > 0,
    )
    bin_centers = (np.arange(N_X_BINS, dtype=np.float64) + 0.5) / N_X_BINS

    summary = {
        "phase_time": np.asarray(phase_times, dtype=np.float64),
        "order_parameter": np.asarray(order_values, dtype=np.float64),
        "volume": np.asarray(volumes, dtype=np.float64),
        "bin_centers": bin_centers,
        "mean_a_fraction": mean_a_fraction,
        "n_frames": len(phase_times),
    }

    np.savez_compressed(
        cache_path,
        phase_time=summary["phase_time"],
        order_parameter=summary["order_parameter"],
        volume=summary["volume"],
        bin_centers=summary["bin_centers"],
        mean_a_fraction=summary["mean_a_fraction"],
        n_frames=np.asarray([summary["n_frames"]], dtype=np.int64),
    )

    return summary


temperature_values = shared_temperature_values(RESULT_SPECS)
if not temperature_values:
    raise RuntimeError("No shared temperature folders were found across the three result roots")

print("Shared temperatures:", temperature_values)

analysis_by_spec: dict[str, dict[float, dict]] = {spec["id"]: {} for spec in RESULT_SPECS}
temp_dirs_by_spec = {spec["id"]: list_temperature_dirs(spec) for spec in RESULT_SPECS}

for spec in RESULT_SPECS:
    print(f"\n== {spec['label']} ==")
    for temp_value in temperature_values:
        temp_dir = temp_dirs_by_spec[spec["id"]][temp_value]
        print(f"Processing {temp_dir.relative_to(REPO_ROOT)}")
        summary = analyze_temperature(spec, temp_dir)
        analysis_by_spec[spec["id"]][temp_value] = summary
        remaining_missing = len(missing_snapshot_paths(spec, temp_dir))
        print(
            f"  second-phase frames={summary['n_frames']}, "
            f"missing snapshot plots after pass={remaining_missing}"
        )

Shared temperatures: [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

== 20260309 NPH ==
Processing results/20260309_NPH_batch_xyz_saving/T=0.5
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260309_NPH_batch_xyz_saving/T=0.6
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260309_NPH_batch_xyz_saving/T=0.7
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260309_NPH_batch_xyz_saving/T=0.8
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260309_NPH_batch_xyz_saving/T=0.9
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260309_NPH_batch_xyz_saving/T=1.0
  second-phase frames=5000, missing snapshot plots after pass=0

== 20260311 NPH piston ==
Processing results/20260311_NPH_batch_xyz_saving_piston/T=0.5
  second-phase frames=5000, missing snapshot plots after pass=0
Processing results/20260311_NPH_batch_xyz_saving_piston/T=0.6

## Plot notes

- The density profile is plotted against reduced coordinate `x / Lx` so the three simulation families can be compared on the same x-axis even when the box width differs.
- Before averaging the second-phase A-fraction profile over time, each frame is circularly shifted so the A-rich slab center aligns at `x / Lx = 0.5`. This avoids washing out the profile when the interface drifts under periodic boundary conditions.
- The “volume” plotted below is the 2D box area `Lx * Ly`, matching the metadata stored in each XYZ frame.

In [8]:
saved_order_paths = []

for temp_value in temperature_values:
    fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=FIG_DPI)
    for spec in RESULT_SPECS:
        summary = analysis_by_spec[spec["id"]][temp_value]
        ax.plot(
            summary["phase_time"],
            summary["order_parameter"],
            label=spec["label"],
            color=spec["color"],
            linewidth=1.6,
        )

    ax.set_title(f"Second-phase order parameter vs time (T={temp_value:g})")
    ax.set_xlabel("second-phase time")
    ax.set_ylabel(r"$\Phi(t)$")
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    output_path = ORDER_DIR / f"order_parameter_second_phase_T_{safe_temp_label(temp_value)}.svg"
    fig.savefig(output_path, dpi=FIG_DPI)
    plt.close(fig)
    saved_order_paths.append(output_path)

print("Saved order-parameter plots:")
for path in saved_order_paths:
    print(" -", path.relative_to(REPO_ROOT))

Saved order-parameter plots:
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_0p5.svg
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_0p6.svg
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_0p7.svg
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_0p8.svg
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_0p9.svg
 - results/analysis_NPH_xyz_batches/order_parameter/order_parameter_second_phase_T_1p0.svg


In [9]:
saved_profile_paths = []

for temp_value in temperature_values:
    fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=FIG_DPI)
    for spec in RESULT_SPECS:
        summary = analysis_by_spec[spec["id"]][temp_value]
        ax.plot(
            summary["bin_centers"],
            summary["mean_a_fraction"],
            label=spec["label"],
            color=spec["color"],
            linewidth=1.8,
        )

    ax.set_title(f"Aligned second-phase A-fraction profile along x/Lx (T={temp_value:g})")
    ax.set_xlabel("reduced position x / Lx")
    ax.set_ylabel("A fraction (1 = all A, 0 = all B)")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    output_path = PROFILE_DIR / f"a_fraction_second_phase_T_{safe_temp_label(temp_value)}.svg"
    fig.savefig(output_path, dpi=FIG_DPI)
    plt.close(fig)
    saved_profile_paths.append(output_path)

print("Saved density-profile plots:")
for path in saved_profile_paths:
    print(" -", path.relative_to(REPO_ROOT))

Saved density-profile plots:
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_0p5.svg
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_0p6.svg
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_0p7.svg
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_0p8.svg
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_0p9.svg
 - results/analysis_NPH_xyz_batches/density_x/a_fraction_second_phase_T_1p0.svg


In [10]:
saved_volume_paths = []
volume_specs = [spec for spec in RESULT_SPECS if spec["has_volume"]]

for temp_value in temperature_values:
    fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=FIG_DPI)
    for spec in volume_specs:
        summary = analysis_by_spec[spec["id"]][temp_value]
        ax.plot(
            summary["phase_time"],
            summary["volume"],
            label=spec["label"],
            color=spec["color"],
            linewidth=1.7,
        )

    ax.set_title(f"Second-phase volume = Lx × Ly vs time (T={temp_value:g})")
    ax.set_xlabel("second-phase time")
    ax.set_ylabel("volume (Lx × Ly)")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    output_path = VOLUME_DIR / f"volume_second_phase_T_{safe_temp_label(temp_value)}.svg"
    fig.savefig(output_path, dpi=FIG_DPI)
    plt.close(fig)
    saved_volume_paths.append(output_path)

print("Saved volume plots:")
for path in saved_volume_paths:
    print(" -", path.relative_to(REPO_ROOT))

Saved volume plots:
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_0p5.svg
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_0p6.svg
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_0p7.svg
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_0p8.svg
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_0p9.svg
 - results/analysis_NPH_xyz_batches/volume/volume_second_phase_T_1p0.svg
